In [230]:
import pandas as pd

In [231]:
df=pd.read_csv("/Users/vijaypatidar/vijay/Anshul/deliver time prediction/data/Food_Delivery_1.csv")
df.sample(5)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
41236,0x26ae,PUNERES20DEL02,23.0,4.8,18.592718,73.773572,18.672718,73.853572,08-03-2022,21:52:00,21:58:04,Sandstorms,Jam,2,Meal,electric_scooter,1.0,No,Urban,21
33908,0x1dd6,RANCHIRES18DEL03,35.0,4.9,23.351489,85.324253,23.381489,85.354253,19-03-2022,17:12:08,17:24:59,Stormy,Medium,1,Snack,motorcycle,0.0,No,Urban,29
5645,0x5d89,VADRES010DEL03,31.0,4.9,22.310329,73.169083,22.370329,73.229083,03-03-2022,21:26:32,21:36:23,Windy,Jam,0,Drinks,motorcycle,0.0,No,Urban,31
22544,0xb67,SURRES17DEL02,25.0,4.5,21.149569,72.772697,21.169569,72.792697,19-03-2022,8:25:59,8:32:03,Stormy,Low,2,Buffet,motorcycle,0.0,No,Metropolitian,13
36602,0x733b,CHENRES14DEL01,31.0,4.2,13.026279,80.174568,13.066279,80.214568,11-03-2022,14:36:29,14:54:05,Windy,High,0,Meal,motorcycle,2.0,Yes,Metropolitian,43


In [232]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in KM

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [233]:
df["Delivery_Distance"] = df.apply(
    lambda row: haversine_distance(
        row["Restaurant_latitude"],
        row["Restaurant_longitude"],
        row["Delivery_location_latitude"],
        row["Delivery_location_longitude"]
    ),
    axis=1
)

In [234]:
df["Delivery_Distance"].head()

0     3.025149
1    20.183530
2     1.552758
3     7.790401
4     6.210138
Name: Delivery_Distance, dtype: float64

In [235]:
df["Delivery_Distance"].describe()

count    41953.000000
mean         9.720284
std          5.603083
min          1.465067
25%          4.657655
50%          9.193021
75%         13.680920
max         20.969489
Name: Delivery_Distance, dtype: float64

In [236]:
df[
    [
        "Time_Orderd",
        "Time_Order_picked"
    ]
].dtypes

Time_Orderd          object
Time_Order_picked    object
dtype: object

In [237]:
df["Time_Orderd"] = pd.to_datetime(
    df["Time_Orderd"],
    format="%H:%M:%S",
    errors="coerce"
)

df["Time_Order_picked"] = pd.to_datetime(
    df["Time_Order_picked"],
    format="%H:%M:%S",
    errors="coerce"
)

In [238]:
df["Preparation_Time"] = (
    df["Time_Order_picked"] -
    df["Time_Orderd"]
).dt.total_seconds() / 60

df.loc[
    df["Preparation_Time"] < 0,
    "Preparation_Time"
] += 24 * 60

In [239]:
df[
    [
        "Time_Orderd",
        "Time_Order_picked",
        "Preparation_Time"
    ]
].head()

,Time_Orderd,Time_Order_picked,Preparation_Time
0,1900-01-01 11:33:33,1900-01-01 11:45:29,11.933333
1,1900-01-01 19:45:37,1900-01-01 19:51:49,6.200000
2,1900-01-01 08:32:58,1900-01-01 08:48:47,15.816667
3,1900-01-01 18:03:58,1900-01-01 18:12:52,8.900000
4,1900-01-01 13:34:16,1900-01-01 13:45:36,11.333333


In [240]:
# # Production Feature Set
# X_production = df.drop(
#     columns=[
#         "Preparation_Time",
#         "Time_taken(min)"
#     ]
# )

# # Experimental Feature Set
# X_experiment = df.drop(
#     columns=[
#         "Time_taken(min)"
#     ]
# )

In [241]:
df["Order_Date"].dtype

dtype('O')

In [242]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)
df["Order_Date"].head()

0   2022-03-19
1   2022-03-25
2   2022-03-19
3   2022-04-05
4   2022-03-26
Name: Order_Date, dtype: datetime64[ns]

In [243]:
df["Order_Hour"] = df["Time_Orderd"].dt.hour
df["Order_Hour"].sample(4)

26107     8.0
19955    19.0
17028    17.0
28302    18.0
Name: Order_Hour, dtype: float64

In [244]:
df["Order_Minute"] = df["Time_Orderd"].dt.minute
df["Order_Minute"].head()

0    33.0
1    45.0
2    32.0
3     3.0
4    34.0
Name: Order_Minute, dtype: float64

In [245]:
df["Day_of_Week"] = df["Order_Date"].dt.day_name()
df['Day_of_Week'].head()

0    Saturday
1      Friday
2    Saturday
3     Tuesday
4    Saturday
Name: Day_of_Week, dtype: object

In [246]:
df["Month"] = df["Order_Date"].dt.month
df['Month'].head()

0    3
1    3
2    3
3    4
4    3
Name: Month, dtype: int64

In [247]:
df["Is_Weekend"] = (
    df["Order_Date"]
      .dt.dayofweek
      .isin([5, 6])
      .astype(int)
)
df["Is_Weekend"].head()

0    1
1    0
2    1
3    0
4    1
Name: Is_Weekend, dtype: int64

In [248]:
df[
    [
        "Order_Date",
        "Time_Orderd",
        "Order_Hour",
        "Order_Minute",
        "Day_of_Week",
        "Month",
        "Is_Weekend"
    ]
].head()

,Order_Date,Time_Orderd,Order_Hour,Order_Minute,Day_of_Week,Month,Is_Weekend
0,2022-03-19,1900-01-01 11:33:33,11.0,33.0,Saturday,3,1
1,2022-03-25,1900-01-01 19:45:37,19.0,45.0,Friday,3,0
2,2022-03-19,1900-01-01 08:32:58,8.0,32.0,Saturday,3,1
3,2022-04-05,1900-01-01 18:03:58,18.0,3.0,Tuesday,4,0
4,2022-03-26,1900-01-01 13:34:16,13.0,34.0,Saturday,3,1


In [249]:
df_model.sample(5)

,Delivery_person_Age,Delivery_person_Ratings,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min),Delivery_Distance,Order_Hour,Order_Minute,Day_of_Week,Month,Is_Weekend
10142,21.0,4.9,conditions Fog,Low,0,Snack,motorcycle,1.0,No,Metropolitian,(min) 21,17.098754,23.0,36.0,Thursday,3,0
18405,27.0,4.7,conditions Windy,Jam,1,Buffet,scooter,1.0,No,Metropolitian,(min) 28,10.756384,19.0,21.0,Friday,3,0
31692,23.0,4.9,conditions Sunny,Low,0,Meal,motorcycle,0.0,No,Urban,(min) 25,13.631449,22.0,46.0,Friday,3,0
37292,36.0,4.8,conditions Stormy,High,1,Buffet,motorcycle,0.0,No,Metropolitian,(min) 25,5.958832,14.0,38.0,Thursday,3,0
121,31.0,4.6,conditions Cloudy,Jam,0,Snack,motorcycle,0.0,No,Metropolitian,(min) 25,9.087973,21.0,21.0,Thursday,3,0


In [250]:
drop_columns = [
    "ID",
    "Delivery_person_ID",

    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",

    "Order_Date",
    "Time_Orderd",
    "Time_Order_picked",

    "Preparation_Time"
]

In [251]:
df_model=df.drop(columns=drop_columns)

In [252]:
df_model.sample(5)

,Delivery_person_Age,Delivery_person_Ratings,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min),Delivery_Distance,Order_Hour,Order_Minute,Day_of_Week,Month,Is_Weekend
8292,31.0,4.9,Cloudy,Medium,0,Drinks,motorcycle,1.0,No,Urban,27,7.683191,17.0,51.0,Sunday,3,1
26847,39.0,5.0,Sunny,Jam,0,Drinks,motorcycle,1.0,No,Metropolitian,23,4.589099,20.0,52.0,Saturday,3,1
37465,32.0,5.0,Sunny,Low,2,Snack,motorcycle,1.0,No,Metropolitian,17,9.313991,22.0,33.0,Thursday,3,0
40858,30.0,4.4,Fog,Low,1,Meal,motorcycle,2.0,No,Metropolitian,39,12.161605,0.0,1.0,Wednesday,4,0
7165,30.0,4.7,Cloudy,Low,0,Drinks,motorcycle,1.0,No,Metropolitian,29,7.660952,23.0,47.0,Saturday,3,1


In [253]:
df_model.shape

(41953, 17)

In [254]:
df_model.columns.tolist()

['Delivery_person_Age',
 'Delivery_person_Ratings',
 'Weatherconditions',
 'Road_traffic_density',
 'Vehicle_condition',
 'Type_of_order',
 'Type_of_vehicle',
 'multiple_deliveries',
 'Festival',
 'City',
 'Time_taken(min)',
 'Delivery_Distance',
 'Order_Hour',
 'Order_Minute',
 'Day_of_Week',
 'Month',
 'Is_Weekend']

In [255]:
df_model.to_csv(
    "../data/food_delivery_processed.csv",
    index=False
)